[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C77_Video_World_Models_Course/05_video_eval/05_video_eval.ipynb)

# C77 模块 05 · 视频评测

三个病理，每个都精确：

1. **同分布之间的 FD 恒为正**且 $\propto 1/N$（$N$ 跨 64 倍时系数稳定在 0.577–0.583）；
2. 对**匹配前两阶矩**的分布**看不出差别**（0.000587 vs 同分布基线 0.000603）；
3. **逐帧特征对帧序的敏感度是 $7.1\times10^{-15}$**（机器精度）。

纯 numpy / CPU / 离线，不训练任何网络。

In [ ]:
import numpy as np

def frechet(m1, C1, m2, C2):
    '''高斯假设下的 Frechet 距离（= FID/FVD 的定义）。'''
    d = m1 - m2
    ev1, V1 = np.linalg.eigh(C1)
    S1 = V1 @ np.diag(np.sqrt(np.clip(ev1, 0, None))) @ V1.T
    ev = np.clip(np.linalg.eigvalsh(S1 @ C2 @ S1), 0, None)
    return float(d @ d + np.trace(C1) + np.trace(C2) - 2*np.sum(np.sqrt(ev)))

def fd_samples(X, Y):
    '''从两组样本估 Frechet 距离。'''
    return frechet(X.mean(0), np.cov(X, rowvar=False),
                   Y.mean(0), np.cov(Y, rowvar=False))

print('FVD = FID 的直接搬运：把视频过一个时空特征提取器，然后算 Frechet 距离。')
print('所以它继承了 FID 的全部性质 —— 包括本模块要量的三个病理。')

## 1. 病理一：同分布之间的 FD 不是零，且 $\propto 1/N$

In [ ]:
d = 32
print(f'特征维 d={d}，两组都来自标准正态。真值应为 0')
print()
print('     N      FD 均值      FD 中位数    d²/N        实测/(d²/N)')
null_fd = {}
for N in (64, 128, 256, 512, 1024, 4096):
    vals = []
    for s in range(200):
        r = np.random.default_rng(s)
        vals.append(fd_samples(r.normal(0, 1, (N, d)), r.normal(0, 1, (N, d))))
    vals = np.array(vals)
    null_fd[N] = float(vals.mean())
    print(f'  {N:6d}   {vals.mean():10.4f}   {np.median(vals):10.4f}   '
          f'{d*d/N:9.3f}   {vals.mean()/(d*d/N):12.3f}')

# 1/N 律：ratio 稳定
ratios = [null_fd[N] / (d*d/N) for N in null_fd]
assert max(ratios) / min(ratios) < 1.05, \
    f'1/N 律应精确（比值应稳定）：{[round(r,4) for r in ratios]}'
# FD 恒为正
assert all(v > 0 for v in null_fd.values()), '同分布下 FD 应恒为正'
# N 翻倍 FD 减半
for N in (64, 128, 256, 512):
    assert abs(null_fd[N] / null_fd[2*N] - 2.0) < 0.15, \
        f'N 从 {N} 到 {2*N}: FD 应减半，实测 {null_fd[N]/null_fd[2*N]:.3f}'
assert abs(null_fd[1024] / null_fd[4096] - 4.0) < 0.6, \
    f'N 从 1024 到 4096（4 倍）: FD 应降 4 倍，实测 {null_fd[1024]/null_fd[4096]:.3f}'

print()
print(f'✅ 1/N 律精确：N 跨 64 倍，实测/(d²/N) 稳定在 '
      f'{min(ratios):.3f}–{max(ratios):.3f}')
print(f'✅ 而 N=64 时零假设值已经是 {null_fd[64]:.2f}')
print()
print('系数对特征维 d 的依赖:')
print('    d      N=256      N=1024     FD/(d²/N) @256   @1024')
for dd in (8, 16, 32, 64):
    row = {}
    for N in (256, 1024):
        v = np.mean([fd_samples(np.random.default_rng(s).normal(0,1,(N,dd)),
                                np.random.default_rng(1000+s).normal(0,1,(N,dd)))
                     for s in range(120)])
        row[N] = v
    print(f'  {dd:5d}   {row[256]:9.4f}  {row[1024]:9.4f}   '
          f'{row[256]/(dd*dd/256):14.3f}   {row[1024]/(dd*dd/1024):7.3f}')
print()
print('-> 1/N 律对每个 d 都精确（同一 d 下两列的系数一致），')
print('   而**系数本身随 d 变化**（0.85 附近降到 0.54 附近）。')
print('   所以可断言的是 1/N 律；d 依赖只能报实测系数。')
print()
print('⚠️  最常见的口径错误：**不同 N 下的 FVD 不可比**。')
print(f'   「我们 FVD=180，基线 210」—— 若两者 N 不同，这个差可能全由有限样本偏差解释。')
print(f'   而视频评测里 N=64 很常见（生成一段视频很贵），此时 d=32 的零假设值已是 '
      f'{null_fd[64]:.2f}。')

## 2. 病理二：只看前两阶矩

构造两个**均值与协方差完全相同**但形状截然不同的分布。

In [ ]:
r = np.random.default_rng(0)
N2, D2 = 100_000, 8

# A：第 0 维是标准正态
A_feat = r.normal(0, 1, (N2, D2))
# B：第 0 维是 ±1 的两点分布（方差同为 1），其余维与 A 同分布
B_feat = r.normal(0, 1, (N2, D2))
B_feat[:, 0] = r.choice([-1.0, 1.0], N2) + r.normal(0, 1e-6, N2)

def kurt(x):
    return float(((x - x.mean())**4).mean() / x.var()**2)

print('A = 第 0 维标准正态；B = 第 0 维是 ±1 的两点分布（方差同为 1）')
print(f'  A: 均值[0] = {A_feat[:,0].mean():+.5f}, 方差[0] = {A_feat[:,0].var():.5f}, '
      f'峰度[0] = {kurt(A_feat[:,0]):.4f}（正态 = 3）')
print(f'  B: 均值[0] = {B_feat[:,0].mean():+.5f}, 方差[0] = {B_feat[:,0].var():.5f}, '
      f'峰度[0] = {kurt(B_feat[:,0]):.4f}（两点 = 1）')
print()
fd_ab = fd_samples(A_feat, B_feat)
fd_null = fd_samples(A_feat, r.normal(0, 1, (N2, D2)))
print(f'  A 与 B 的 Frechet 距离        = {fd_ab:.6f}')
print(f'  同分布 N={N2} 的基线          = {fd_null:.6f}')
print(f'  比值                          = {fd_ab/fd_null:.4f}')

assert abs(A_feat[:,0].var() - B_feat[:,0].var()) < 0.01, '方差应匹配'
assert abs(kurt(A_feat[:,0]) - 3.0) < 0.1 and abs(kurt(B_feat[:,0]) - 1.0) < 0.1, \
    '峰度应差别巨大'
assert fd_ab < 3 * fd_null, \
    f'FD 应与同分布基线同量级（{fd_ab:.6f} vs {fd_null:.6f}）'

print()
print(f'✅ 峰度 {kurt(A_feat[:,0]):.2f} vs {kurt(B_feat[:,0]):.2f}（一个连续单峰、一个只有两个取值）')
print(f'✅ 而 FD 只有 {fd_ab:.6f}，与同分布基线 {fd_null:.6f} 同量级')
print()
print('-> 这不是精度问题，是**定义**：Frechet 距离的公式里只出现 μ 与 Σ。')
print('   所以任何两个同均值同协方差的分布，FVD 都认为它们相同。')
print()
print('   直接对应的真实失效：**模式坍缩到几个离散模式** ——')
print('   生成器只会产出少数几种运动模式，而只要它们的混合的一二阶矩对得上，')
print('   FVD 就看不出来。这是视频生成最常见的失效之一。')

## 3. 病理三：特征决定一切

同一批视频，**只把帧顺序打乱**，看指标变不变。

In [ ]:
def make_videos(n=3000, T=8, d=16, seed=1, corr=0.9):
    '''合成「视频特征序列」：每帧 d 维，时间上做 AR(1)。'''
    r = np.random.default_rng(seed)
    V = r.normal(0, 1, (n, T, d))
    for t in range(1, T):
        V[:, t] = corr*V[:, t-1] + np.sqrt(1-corr**2)*V[:, t]
    return V

def feats_perframe(V):
    '''逐帧特征后对时间取平均 —— 很多实现就是这么做的。'''
    return V.mean(1)
def feats_spatiotemporal(V):
    '''加上时间差分的最简时空特征。'''
    return np.concatenate([V.mean(1), np.abs(np.diff(V, axis=1)).mean(1)], 1)

V = make_videos()
perm = np.random.default_rng(2).permutation(V.shape[1])
Vs = V[:, perm, :]
print(f'同一批 {len(V)} 段视频，只把帧顺序打乱（perm = {perm}）')
print()
fd_pf = fd_samples(feats_perframe(V), feats_perframe(Vs))
fd_st = fd_samples(feats_spatiotemporal(V), feats_spatiotemporal(Vs))
print(f'  逐帧特征取时间平均:    FD = {fd_pf:.6e}')
print(f'  含时间差分的时空特征:  FD = {fd_st:.6e}')
assert fd_pf < 1e-10, f'逐帧平均特征应对帧序**恒等于**免疫，得到 {fd_pf:.3e}'
assert fd_st > 0.1, f'时空特征应能看出帧序，得到 {fd_st:.3e}'

print()
print(f'✅ {fd_pf:.1e} 不是「不够敏感」，是**恒等于零**：')
print('   对时间取平均之后，帧的顺序在特征里已经不存在了。')
print()
print('   所以「用逐帧特征算出来的 FVD」根本不是在评测时间维 ——')
print('   它是一个逐帧的 FID，只是名字里有个 V。')

# 多个置换都一样
print()
print('  10 个不同的随机置换:')
fds = []
for s in range(10):
    p = np.random.default_rng(100+s).permutation(V.shape[1])
    fds.append(fd_samples(feats_perframe(V), feats_perframe(V[:, p, :])))
print(f'    逐帧特征: 最大 {max(fds):.3e}，全部 < 1e-10: {all(v < 1e-10 for v in fds)}')
assert all(v < 1e-10 for v in fds)
print('    -> 与置换无关，恒为机器精度。')

## ✏️ 练习 1：$\text{FVD}_\infty$——把有限样本偏差外推掉

既然零假设值 $\propto 1/N$，就可以在若干个 $N$ 上测，然后线性外推到 $1/N = 0$。

实现 `fd_infinity(X, Y, Ns, reps)`：在给定的若干个子样本量上估 FD，
对 $1/N$ 做线性回归，返回截距（$= \text{FD}_\infty$）。

In [ ]:
def fd_infinity(X, Y, Ns=(64, 128, 256, 512), reps=20, seed=0):
    '''FD_inf：在若干子样本量上估 FD，对 1/N 线性外推到 1/N = 0。

    参数
    ----
    X, Y : 两组特征样本
    Ns   : 子样本量（必须都 <= min(len(X), len(Y))）
    reps : 每个 N 重采样多少次取平均
    seed : 重采样 seed

    返回
    ----
    (fd_inf, slope, curve) :
      fd_inf = 对 1/N 线性回归的截距
      slope  = 斜率
      curve  = {N: 该 N 下的 FD 均值}
    '''
    r = np.random.default_rng(seed)
    # TODO: 对每个 N，重采样 reps 次（各取 N 个样本），算 fd_samples 的均值；
    #       然后用 np.polyfit(1/N, FD, 1) 得斜率与截距；返回 (截距, 斜率, 曲线)
    raise NotImplementedError

In [ ]:
# 自测
_d = 32
_r = np.random.default_rng(7)
_NS = (64, 128, 256, 512, 1024)

print('  (a) 同分布：FD_inf 应接近 0，而各 N 的 FD 都明显 > 0')
_X = np.random.default_rng(7).normal(0, 1, (4000, _d))
_Y = np.random.default_rng(8).normal(0, 1, (4000, _d))
_inf, _sl, _cv = fd_infinity(_X, _Y, Ns=_NS)
print('     N      FD')
for _N in _NS:
    print(f'   {_N:5d}   {_cv[_N]:.4f}')
print(f'    FD_inf（外推）= {_inf:+.5f}   斜率 = {_sl:.2f}')
assert abs(_inf) < 0.05, f'同分布的 FD_inf 应接近 0，得到 {_inf:.5f}'
assert _cv[_NS[0]] > 5.0, f'N=64 时的 FD 应明显 >0，得到 {_cv[_NS[0]]:.4f}'
print(f'    -> 外推把偏差从 {_cv[_NS[0]]:.2f}（N=64）降到 {abs(_inf):.5f}'
      f'（{_cv[_NS[0]]/max(abs(_inf),1e-9):.0f} 倍）')

print()
print('  (b) 真有差别时，FD_inf 保留那个差别 —— 但精度是**绝对**的，不是相对的')
print('    均值偏移   真 FD     N=64 的 FD   FD_inf      绝对残差   相对残差')
_abs_res = []
for _shift in (0.0, 0.1, 0.3, 0.5, 1.0, 2.0):
    _Y2 = np.random.default_rng(8).normal(0, 1, (4000, _d))   # 固定 seed 便于复现
    _Y2[:, 0] += _shift
    _i2, _, _c2 = fd_infinity(_X, _Y2, Ns=_NS)
    _true = _shift**2
    _abs_res.append(abs(_i2 - _true))
    print(f'    {_shift:8.1f}   {_true:7.4f}   {_c2[64]:10.4f}   {_i2:+9.4f}   '
          f'{abs(_i2-_true):8.4f}   {abs(_i2-_true)/max(_true,0.02)*100:7.1f}%')

# 绝对残差有界，而相对残差在小效应上很大
assert max(_abs_res) < 0.10, \
    f'绝对残差应 <0.10，实测最大 {max(_abs_res):.4f}'
_Y_small = np.random.default_rng(8).normal(0, 1, (4000, _d)); _Y_small[:, 0] += 0.1
_i_small = fd_infinity(_X, _Y_small, Ns=_NS)[0]
print()
print(f'    ⚠️  偏移 0.1（真 FD = 0.0100）时 FD_inf = {_i_small:+.4f}'
      f' —— **{_i_small/0.01:.1f} 倍高估**')
print('        残差的结构是「绝对底 + 小的乘性项」：')
print(f'        真 FD 从 0 到 4 时残差从 {_abs_res[0]:.4f} 涨到 {_abs_res[-1]:.4f}，')
print(f'        而**相对**残差从（无定义）降到 {_abs_res[-1]/4.0*100:.1f}%。')
print(f'    -> 所以 FD_inf 在大效应上相对精度很好（>= 0.09 时相对残差 <25%），')
print(f'       而在小效应上不可靠（绝对底约 {_abs_res[0]:.3f}）。')
print('       注意 1/N 线性拟合**不保证非负**，所以极小效应上它甚至可能给出负值。')

print()
print('  (c) 而**固定 N** 的 FD 会把小差别完全埋掉:')
_c_same = fd_infinity(_X, np.random.default_rng(8).normal(0, 1, (4000, _d)), Ns=_NS)[2][64]
for _shift in (0.0, 0.1, 0.3, 1.0):
    _Y2 = np.random.default_rng(8).normal(0, 1, (4000, _d))
    _Y2[:, 0] += _shift
    _c2 = fd_infinity(_X, _Y2, Ns=_NS)[2][64]
    print(f'    偏移 {_shift:.1f}: N=64 的 FD = {_c2:8.4f}'
          f'（真值 {_shift**2:.4f}，与「同分布」的 {_c_same:.4f} 相比差 '
          f'{abs(_c2-_c_same):.4f}）')
_Yb = np.random.default_rng(8).normal(0, 1, (4000, _d)); _Yb[:, 0] += 0.1
_c_b = fd_infinity(_X, _Yb, Ns=_NS)[2][64]
assert abs(_c_b - _c_same) < _c_same, \
    f'小差别应被有限样本偏差淹没（差 {abs(_c_b-_c_same):.3f} vs 偏差 {_c_same:.3f}）'
print(f'    -> 偏移 0.1 造成的差 {abs(_c_b-_c_same):.3f} 远小于偏差本身 {_c_same:.3f}。')

print()
print(f'✅ 同分布的 FD_inf = {_inf:+.5f}（把 N=64 的偏差降了 '
      f'{_cv[_NS[0]]/max(abs(_inf),1e-9):.0f} 倍）')
print(f'✅ 真有差别时 FD_inf 的绝对残差 <= {max(_abs_res):.3f}（6 个偏移量），')
print(f'   而真 FD >= 0.09 时相对残差 < 25%')
print(f'⚠️  但小效应上不可靠：偏移 0.1（真 FD 0.01）时给出 {_i_small:+.4f}'
      f'（{_i_small/0.01:.1f} 倍高估）')
print()
print('   -> 两条可操作的结论:')
print('      · 「固定 N」是最低要求；要比较**接近**的模型必须做外推。')
print(f'      · 而外推本身有一个绝对精度下限（这里约 {max(_abs_res):.2f}）——')
print('        小于它的效应无论怎么外推都分辨不出，只能加大 N 或换指标。')

## ✏️ 练习 2：一个能看到模式坍缩的指标

FVD 对第 2 节那个例子免疫。实现一个 $k$-NN 覆盖率来补它：

`coverage(X_real, Y_gen, k)`：对每个**真实**样本，看它的第 $k$ 近邻半径内
有没有生成样本；返回被覆盖的真实样本比例。

模式坍缩会让覆盖率下降，而 FVD 看不出来。

In [ ]:
def coverage(X_real, Y_gen, k=5, chunk=512):
    '''k-NN 覆盖率（recall 的一个简单实现）。

    对每个真实样本 x：
      r_k(x) = x 到其**第 k 个真实近邻**的距离（不含自己）
      被覆盖 = 存在生成样本落在 x 的 r_k(x) 半径内
    返回被覆盖的真实样本比例。

    参数
    ----
    X_real : 真实样本 (n, d)
    Y_gen  : 生成样本 (m, d)
    k      : 近邻数
    chunk  : 分块大小（避免建大矩阵）

    返回
    ----
    float : 覆盖率 ∈ [0, 1]
    '''
    # TODO: ① 对 X_real 内部算第 k 近邻距离 r_k（用 np.partition，排除自身距离 0）
    #       ② 算每个真实样本到最近**生成**样本的距离 dmin
    #       ③ 返回 mean(dmin <= r_k)
    #       两步都要分块，避免 (n, n) / (n, m) 的大矩阵
    raise NotImplementedError

In [ ]:
# 自测
_r2 = np.random.default_rng(11)
_n, _dd = 4000, 8

print('  (a) 同分布：覆盖率应高')
_XR = _r2.normal(0, 1, (_n, _dd))
_YG = _r2.normal(0, 1, (_n, _dd))
_cov_same = coverage(_XR, _YG, k=5)
print(f'    覆盖率 = {_cov_same:.4f}')
assert _cov_same > 0.7, f'同分布的覆盖率应 >0.7，得到 {_cov_same:.4f}'

print()
print('  (b) 第 2 节那个例子：FVD 看不出，覆盖率能看出')
_A2 = _r2.normal(0, 1, (_n, _dd))
_B2 = _r2.normal(0, 1, (_n, _dd))
_B2[:, 0] = _r2.choice([-1.0, 1.0], _n)            # 第 0 维坍缩成两点
_fd = fd_samples(_A2, _B2)
_fd_base = fd_samples(_A2, _r2.normal(0, 1, (_n, _dd)))
_cov_collapse = coverage(_A2, _B2, k=5)
print(f'    FVD:     A vs B = {_fd:.5f}，同分布基线 = {_fd_base:.5f}'
      f'（比值 {_fd/_fd_base:.2f}）')
print(f'    覆盖率:  A vs B = {_cov_collapse:.4f}，同分布 = {_cov_same:.4f}'
      f'（降低 {(1-_cov_collapse/_cov_same)*100:.0f}%）')
assert _fd < 5 * _fd_base, 'FVD 应看不出差别'
assert _cov_collapse < 0.95 * _cov_same, \
    f'覆盖率应下降（{_cov_collapse:.4f} vs {_cov_same:.4f}）'

print()
print('  (c) 坍缩程度 vs 两个指标')
print('    保留的模式数   FVD / 基线    覆盖率    覆盖率 / 同分布')
for _nmode in (2, 4, 8, 32, 1000):
    _Bm = _r2.normal(0, 1, (_n, _dd))
    _centers = _r2.normal(0, 1, (_nmode,))
    _centers = (_centers - _centers.mean()) / _centers.std()      # 匹配一二阶矩
    _Bm[:, 0] = _centers[_r2.integers(0, _nmode, _n)]
    _f = fd_samples(_A2, _Bm) / _fd_base
    _c = coverage(_A2, _Bm, k=5)
    print(f'    {_nmode:12d}   {_f:10.2f}    {_c:.4f}   {_c/_cov_same:.4f}')

print()
print(f'✅ 同分布覆盖率 {_cov_same:.3f}；坍缩到两点时降到 {_cov_collapse:.3f}')
print(f'✅ 而同一对分布的 FVD 只有基线的 {_fd/_fd_base:.2f} 倍 —— 完全看不出')
print()
print('   -> 覆盖率与 FVD 是**互补**的，不是替代关系：')
print('      FVD 对整体的一二阶矩敏感、对模式结构免疫；')
print('      覆盖率对模式结构敏感、对整体平移不敏感。')
print('      两个都报才能同时管住「分布错位」与「模式坍缩」。')

## ✏️ 练习 3：$N$ 多小算太小

正文说 $N{=}64$ 时零假设值已经是 $9.33$（$d{=}32$）。
把它变成一条可以用来定 $N$ 的规则：实现 `min_N_for_effect(d, target_fd, ratio)`，
返回让「有限样本偏差 $\leq$ 目标效应 / ratio」所需的最小 $N$。

In [ ]:
def null_fd_estimate(d, N, reps=40, seed=0):
    '''同分布下 FD 的均值（有限样本偏差的大小）。'''
    vals = []
    for s in range(reps):
        r = np.random.default_rng(seed + s)
        vals.append(fd_samples(r.normal(0, 1, (N, d)), r.normal(0, 1, (N, d))))
    return float(np.mean(vals))

def min_N_for_effect(d, target_fd, ratio=5.0, Ns=(64, 128, 256, 512, 1024, 2048, 4096)):
    '''让有限样本偏差 <= target_fd / ratio 所需的最小 N。

    用 1/N 律外推而不是逐个测（逐个测太慢）：
      先在一个基准 N0 上测出偏差 b0，则 b(N) ≈ b0 * N0 / N。

    参数
    ----
    d         : 特征维
    target_fd : 你想分辨的效应大小
    ratio     : 要求偏差比效应小多少倍
    Ns        : 候选 N

    返回
    ----
    (min_N, b0, curve) :
      min_N = 最小可行的 N（候选内都不够时返回 None）
      b0    = 在 Ns[0] 上实测的偏差
      curve = {N: 外推的偏差}
    '''
    N0 = Ns[0]
    b0 = null_fd_estimate(d, N0)
    # TODO: 用 b(N) = b0 * N0 / N 外推；返回第一个满足 b(N) <= target_fd/ratio 的 N
    raise NotImplementedError

In [ ]:
# 自测
print('  (a) 外推的偏差与实测吻合（d=32）')
_b0 = null_fd_estimate(32, 64)
print('     N     外推 b0*64/N    实测        相对差')
for _N in (128, 256, 512, 1024):
    _ex = _b0 * 64 / _N
    _me = null_fd_estimate(32, _N)
    print(f'   {_N:5d}   {_ex:12.4f}   {_me:9.4f}   {abs(_ex-_me)/_me*100:6.1f}%')
    assert abs(_ex - _me) / _me < 0.10, f'N={_N}: 外推应与实测吻合到 10%'

print()
print('  (b) 想分辨多小的效应，就需要多大的 N（d=32, ratio=5）')
print('    目标效应   所需最小 N     该 N 下的偏差')
for _tf in (100.0, 20.0, 5.0, 1.0, 0.2):
    _mn, _b, _cv2 = min_N_for_effect(32, _tf, ratio=5.0)
    _shown = str(_mn) if _mn else '>4096'
    _bias = _cv2[_mn] if _mn else float('nan')
    print(f'    {_tf:8.1f}   {_shown:>10s}     {_bias:.4f}')
    if _mn:
        assert _cv2[_mn] <= _tf / 5.0 + 1e-9

print()
print('  (c) 特征维的影响（目标效应固定为 5.0）')
print('     d     所需最小 N   N=64 时的偏差')
for _dd in (8, 16, 32, 64):
    _mn, _b, _ = min_N_for_effect(_dd, 5.0, ratio=5.0)
    print(f'   {_dd:5d}   {str(_mn) if _mn else ">4096":>10s}   {_b:.4f}')
_mn8 = min_N_for_effect(8, 5.0)[0]
_mn64 = min_N_for_effect(64, 5.0)[0]
assert _mn64 > _mn8, 'd 越大所需 N 越大'

print()
print(f'✅ 1/N 外推与实测吻合到 10% 以内（4 个 N 全对）')
print(f'✅ d=8 时需要 N>={_mn8}，d=64 时需要 N>={_mn64}')
print()
print('   -> 这条规则可以在**做实验之前**用：')
print('      先想清「我要分辨多大的差别」，再算所需的 N。')
print('      如果算出来的 N 生成不起，那这个实验就不该被做成 FVD 比较 ——')
print('      应该换成分维度打分或人评。')

## ✏️ 练习 4：帧序置换检验

正文说这个检验应该对每个视频指标跑一次。实现它，
并在五种特征上测——其中有一个是刻意构造的陷阱。

In [ ]:
def order_sensitivity(feat_fn, n=2000, T=8, dfeat=16, seeds=5, corr=0.9):
    '''特征函数对帧序的敏感度（多次随机置换的 FD 中位数）。

    参数
    ----
    feat_fn : 把 (n, T, d) 的视频批映射到 (n, k) 特征的函数
    n, T, dfeat : 批大小、帧数、每帧特征维
    seeds   : 试多少个随机置换
    corr    : 视频的时间相关性

    返回
    ----
    float : FD 的中位数
    '''
    V = make_videos(n=n, T=T, d=dfeat, seed=11, corr=corr)
    # TODO: 对 s in range(seeds)：用 np.random.default_rng(100+s).permutation(T)
    #       打乱帧序，算 fd_samples(feat_fn(V), feat_fn(V_shuffled))，返回中位数
    raise NotImplementedError

In [ ]:
# 自测
def _f_mean(V):
    '''逐帧特征取时间平均。'''
    return V.mean(1)
def _f_sorted(V):
    '''对时间轴**排序**后再平均 —— 一个刻意构造的陷阱。'''
    return np.sort(V, axis=1).mean(1)
def _f_concat(V):
    '''把所有帧拼起来（保留全部顺序信息）。'''
    return V.reshape(len(V), -1)
def _f_diff(V):
    '''时间差分的绝对值均值。'''
    return np.abs(np.diff(V, axis=1)).mean(1)
def _f_firstlast(V):
    '''首末帧之差。'''
    return V[:, -1] - V[:, 0]

print('  特征                              用了多少帧   帧序敏感度      判定')
_res = {}
for _name, _fn, _uses, _expect in [
        ('逐帧平均 mean_t', _f_mean, '全部 T 帧', 'blind'),
        ('**时间排序后**平均', _f_sorted, '全部 T 帧', 'blind'),
        ('全帧拼接', _f_concat, '全部 T 帧', 'sensitive'),
        ('时间差分均值', _f_diff, '全部 T 帧', 'sensitive'),
        ('首末帧之差', _f_firstlast, '仅 2 帧', 'sensitive')]:
    _v = order_sensitivity(_fn)
    _res[_name] = _v
    _verdict = '❌ 对帧序免疫' if _v < 1e-9 else '✅ 能看出帧序'
    print(f'  {_name:32s}  {_uses:10s}  {_v:.6e}   {_verdict}')
    if _expect == 'blind':
        assert _v < 1e-9, f'{_name} 应对帧序免疫，得到 {_v:.3e}'
    else:
        assert _v > 1e-3, f'{_name} 应能看出帧序，得到 {_v:.3e}'

print()
print('✅ 两类特征被干净地分开：免疫的 < 1e-9（机器精度），敏感的 > 1e-3')
print()
print('⚠️  注意两行的对照:')
print('    「时间排序后平均」**用到了全部 T 帧**（它排序了整段），却对帧序完全免疫；')
print('    「首末帧之差」**只用了 2 帧**，却能看出帧序。')
print()
print('   -> 所以判据不是「用了多少帧」，也不是「网络结构能不能看时间」。')
print('      常见的辩护是「我们的提取器是 3D 卷积/时空 Transformer」——')
print('      但**结构能看时间不代表最终特征里保留了帧序信息**。')
print('      唯一可靠的判据是把置换检验真的跑一遍，而它只需要几行代码。')

# 相关性对敏感度的影响
print()
print('  时间相关性 vs 敏感度（时空特征）:')
print('    corr    敏感度')
for _c in (0.0, 0.3, 0.6, 0.9, 0.99):
    _v = order_sensitivity(_f_diff, corr=_c)
    print(f'    {_c:.2f}    {_v:.6e}')
_v0 = order_sensitivity(_f_diff, corr=0.0)
_v9 = order_sensitivity(_f_diff, corr=0.9)
assert _v9 > _v0 * 10, '相关性越高，打乱帧序造成的差别应越大'
print()
print(f'✅ corr=0 时敏感度 {_v0:.2e}（本来就没有时间结构可打乱）')
print(f'   corr=0.9 时是 {_v9:.2e}（{_v9/max(_v0,1e-12):.0f} 倍）')
print('   -> 置换检验的**功效**取决于素材本身的时间相关性。')
print('      在快剪辑素材上做这个检验会得到假的「通过」—— 检验本身需要合适的数据。')

## 📖 参考答案

In [ ]:
def fd_infinity(X, Y, Ns=(64, 128, 256, 512), reps=20, seed=0):
    '''FD_inf：在若干子样本量上估 FD，对 1/N 线性外推。'''
    r = np.random.default_rng(seed)
    curve = {}
    for N in Ns:
        vals = []
        for _ in range(reps):
            ix = r.choice(len(X), N, replace=False)
            iy = r.choice(len(Y), N, replace=False)
            vals.append(fd_samples(X[ix], Y[iy]))
        curve[N] = float(np.mean(vals))
    xs = np.array([1.0/N for N in Ns])
    ys = np.array([curve[N] for N in Ns])
    slope, intercept = np.polyfit(xs, ys, 1)
    return float(intercept), float(slope), curve

def coverage(X_real, Y_gen, k=5, chunk=512):
    '''k-NN 覆盖率（recall 的一个简单实现）。'''
    n = len(X_real)
    # ① 真实样本内部的第 k 近邻距离
    rk = np.empty(n)
    xn = (X_real * X_real).sum(1)
    for i in range(0, n, chunk):
        blk = X_real[i:i+chunk]
        d2 = (blk*blk).sum(1)[:, None] - 2.0*(blk @ X_real.T) + xn[None, :]
        d2 = np.maximum(d2, 0.0)
        # 排除自身（每行的最小值 0）
        part = np.partition(d2, k, axis=1)[:, :k+1]
        part = np.sort(part, axis=1)[:, 1:k+1]
        rk[i:i+chunk] = np.sqrt(part[:, -1])
    # ② 到最近生成样本的距离
    dmin = np.empty(n)
    yn = (Y_gen * Y_gen).sum(1)
    for i in range(0, n, chunk):
        blk = X_real[i:i+chunk]
        d2 = (blk*blk).sum(1)[:, None] - 2.0*(blk @ Y_gen.T) + yn[None, :]
        dmin[i:i+chunk] = np.sqrt(np.maximum(d2.min(1), 0.0))
    return float(np.mean(dmin <= rk))

def min_N_for_effect(d, target_fd, ratio=5.0, Ns=(64, 128, 256, 512, 1024, 2048, 4096)):
    '''让有限样本偏差 <= target_fd/ratio 所需的最小 N（用 1/N 律外推）。'''
    N0 = Ns[0]
    b0 = null_fd_estimate(d, N0)
    curve = {N: b0 * N0 / N for N in Ns}
    thresh = target_fd / ratio
    for N in Ns:
        if curve[N] <= thresh:
            return N, b0, curve
    return None, b0, curve

def order_sensitivity(feat_fn, n=2000, T=8, dfeat=16, seeds=5, corr=0.9):
    '''特征函数对帧序的敏感度（多次随机置换的 FD 中位数）。'''
    V = make_videos(n=n, T=T, d=dfeat, seed=11, corr=corr)
    vals = []
    for s in range(seeds):
        perm = np.random.default_rng(100 + s).permutation(T)
        vals.append(fd_samples(feat_fn(V), feat_fn(V[:, perm, :])))
    return float(np.median(vals))

print('参考答案已定义。')
print()
print('要点：')
print('  1. FD_inf 的 1/N 外推把 N=64 的偏差降 2 个数量级，而真实差别被保留。')
print('     「固定 N」只是最低要求；比较接近的模型必须外推。')
print('  2. 覆盖率与 FVD **互补**：前者对模式结构敏感，后者对一二阶矩敏感。')
print('  3. 「我要分辨多大的差别」决定所需的 N —— 这个账应该在实验之前算。')
print('  4. 帧序置换检验的判据不是「用了多少帧」：')
print('     「排序后平均」用了全部帧却免疫，「首末帧之差」只用 2 帧却敏感。')

## 🧪 真实工程胶囊：一份视频评测报告的最低配置

下面这段代码把本模块与模块 03 的结论合成一份报告：
它对一批生成视频输出五项，并对每一项标注**它管得住什么、管不住什么**。

关键设计：报告里每一项都带一个「这一项对什么免疫」的字段。
因为本模块的三个病理说明，<strong>不写清免疫范围的指标等于没有量纲</strong>。

In [ ]:
def video_eval_report(gen_feats, real_feats, gen_seqs=None, real_seqs=None,
                      d_hint=None, verbose=True):
    '''视频评测的最低配置报告。

    gen_feats / real_feats : (n, T, d) 的**逐帧**特征（本函数自己做两种聚合）
    gen_seqs / real_seqs   : 可选，1D 序列批（给模块 03 的长程诊断）
    '''
    out = {}
    T = gen_feats.shape[1]

    # ① FVD（两种特征各算一次）——并同时报有限样本偏差
    f_pf_g, f_pf_r = gen_feats.mean(1), real_feats.mean(1)
    f_st_g = np.concatenate([gen_feats.mean(1),
                             np.abs(np.diff(gen_feats, axis=1)).mean(1)], 1)
    f_st_r = np.concatenate([real_feats.mean(1),
                             np.abs(np.diff(real_feats, axis=1)).mean(1)], 1)
    N = min(len(f_pf_g), len(f_pf_r))
    d_eff = d_hint if d_hint is not None else f_st_r.shape[1]
    out['fvd_perframe'] = fd_samples(f_pf_g, f_pf_r)
    out['fvd_spatiotemporal'] = fd_samples(f_st_g, f_st_r)
    out['null_bias'] = null_fd_estimate(min(d_eff, 32), min(N, 512), reps=10)
    out['N'] = N

    # ② 覆盖率
    out['coverage'] = coverage(f_st_r[:min(2000, len(f_st_r))],
                               f_st_g[:min(2000, len(f_st_g))], k=5)

    # ③ 帧序置换检验（对**本报告用的两种特征**各跑一次）
    perm = np.random.default_rng(3).permutation(T)
    out['order_sens_perframe'] = fd_samples(f_pf_r, real_feats[:, perm, :].mean(1))
    _sp = real_feats[:, perm, :]
    out['order_sens_spatiotemporal'] = fd_samples(
        f_st_r, np.concatenate([_sp.mean(1), np.abs(np.diff(_sp, axis=1)).mean(1)], 1))

    # ④ 长程（模块 03）
    if gen_seqs is not None and real_seqs is not None:
        ref_sd = float(np.median([np.std(y) for y in real_seqs]))
        bad = sum(1 for y in gen_seqs
                  if (not np.isfinite(y).all()) or np.abs(y).max() > 10*ref_sd)
        out['collapse_rate'] = bad / len(gen_seqs)
        ok = [y for y in gen_seqs
              if np.isfinite(y).all() and np.abs(y).max() <= 10*ref_sd]
        if ok:
            m_g = float(np.median([np.std(y[len(y)//2:]) for y in ok]))
            m_r = float(np.median([np.std(y[len(y)//2:]) for y in real_seqs]))
            out['energy_ratio'] = m_g / max(m_r, 1e-12)

    if verbose:
        print(f'  样本数 N = {out["N"]}，特征维 = {f_st_r.shape[1]}')
        print('  ' + '-' * 70)
        print(f'  ① FVD（逐帧特征）        = {out["fvd_perframe"]:10.4f}')
        print(f'     免疫：帧序（本模块第 3 节实测 {out["order_sens_perframe"]:.1e}）、'
              f'高阶矩、模式坍缩')
        print(f'  ① FVD（时空特征）        = {out["fvd_spatiotemporal"]:10.4f}')
        print(f'     免疫：高阶矩、模式坍缩。有限样本偏差 ≈ {out["null_bias"]:.4f}'
              f'（N={min(out["N"],512)}, d≈{min(d_eff,32)}）')
        print(f'  ② 覆盖率（k=5）          = {out["coverage"]:10.4f}')
        print(f'     免疫：整体平移（它只看局部近邻结构）')
        print(f'  ③ 帧序置换敏感度')
        print(f'       逐帧特征            = {out["order_sens_perframe"]:.3e}'
              + ('   ⚠️  免疫 —— 这一项**不在**评时间维'
                 if out['order_sens_perframe'] < 1e-9 else ''))
        print(f'       时空特征            = {out["order_sens_spatiotemporal"]:.3e}')
        if 'collapse_rate' in out:
            print(f'  ④ 崩坏率                 = {out["collapse_rate"]*100:9.1f}%')
            print(f'     能量比（未崩坏部分）   = {out.get("energy_ratio", float("nan")):10.4f}')
            print(f'     免疫：单帧质量（它只看长程的二阶统计）')
        print('  ' + '-' * 70)
    return out

# 构造一批「生成」与「真实」的视频特征
_REAL = make_videos(n=3000, T=8, d=16, seed=1, corr=0.9)
print('=== 场景 A：生成分布与真实一致 ===')
_GEN_A = make_videos(n=3000, T=8, d=16, seed=555, corr=0.9)
_rA = video_eval_report(_GEN_A, _REAL)
print()

print('=== 场景 B：时间结构错了（corr 0.9 -> 0.3），但**单帧分布不变** ===')
_GEN_B = make_videos(n=3000, T=8, d=16, seed=555, corr=0.3)
_rB = video_eval_report(_GEN_B, _REAL)
assert _rB['fvd_spatiotemporal'] > 5 * _rA['fvd_spatiotemporal'], \
    '时空特征应能看出时间结构变了'
print()

print('=== 场景 C：**矩匹配**的模式坍缩 ===')
print('    把若干特征维坍缩成 ±σ 的两点分布，而符号按 AR(1)(corr=0.9) 翻转 ——')
print('    于是均值、方差、时间相关性**全部匹配**，只有高阶矩不同（峰度 3 -> 1）。')
_GEN_C = make_videos(n=3000, T=8, d=16, seed=555, corr=0.9)
_rc = np.random.default_rng(4)
_p_flip = (1 - 0.9) / 2
for _j in range(16):                       # 全部 16 维都坍缩
    _sg = np.ones((3000, 8))
    _sg[:, 0] = _rc.choice([-1.0, 1.0], 3000)
    for _t in range(1, 8):
        _sg[:, _t] = _sg[:, _t-1] * ((_rc.random(3000) < _p_flip) * (-2) + 1)
    _GEN_C[:, :, _j] = _sg * _GEN_C[:, :, _j].std()
def _kurt(x):
    return float(((x - x.mean())**4).mean() / x.var()**2)
print(f'    真实的 dim0 峰度 = {_kurt(_REAL[:,:,0].ravel()):.4f}（正态 = 3）')
print(f'    生成的 dim0 峰度 = {_kurt(_GEN_C[:,:,0].ravel()):.4f}（两点 = 1）')
print()
_rC = video_eval_report(_GEN_C, _REAL)
print()

print('=== 三个场景的对照 ===')
print('  场景   FVD(逐帧)  /A      FVD(时空)  /A       覆盖率   抓住问题的是谁')
for _tag, _r, _who in [
        ('A', _rA, '（无问题）'),
        ('B', _rB, 'FVD(时空)'),
        ('C', _rC, 'FVD(时空)')]:
    print(f'  {_tag}     {_r["fvd_perframe"]:9.4f} {_r["fvd_perframe"]/_rA["fvd_perframe"]:6.2f}   '
          f'{_r["fvd_spatiotemporal"]:9.4f} {_r["fvd_spatiotemporal"]/_rA["fvd_spatiotemporal"]:6.2f}    '
          f'{_r["coverage"]:.4f}   {_who}')

_pf_B = _rB['fvd_perframe'] / _rA['fvd_perframe']
_st_B = _rB['fvd_spatiotemporal'] / _rA['fvd_spatiotemporal']
_pf_C = _rC['fvd_perframe'] / _rA['fvd_perframe']
_st_C = _rC['fvd_spatiotemporal'] / _rA['fvd_spatiotemporal']

# 场景 B、C 里 FVD(逐帧) 都盲，而 FVD(时空) 都抓住
assert _st_B > 3 * _pf_B, f'场景 B: 时空应远比逐帧敏感（{_st_B:.2f} vs {_pf_B:.2f}）'
assert _st_C > 5 * _pf_C, f'场景 C: 时空应远比逐帧敏感（{_st_C:.2f} vs {_pf_C:.2f}）'
assert _pf_C < 1.0, f'场景 C 的 FVD(逐帧) 应**低于**基线（完全盲），实测 {_pf_C:.2f}'
assert _st_C > 5.0, f'场景 C 的 FVD(时空) 应明显放大，实测 {_st_C:.2f}'

print()
print('工程含义：')
print(f'  · **两个场景里 FVD(逐帧) 都是盲的**：场景 B 放大 {_pf_B:.2f}x，')
print(f'    场景 C 甚至是 {_pf_C:.2f}x（**低于**同分布基线）。')
print(f'    而 FVD(时空) 分别放大 {_st_B:.1f}x 与 {_st_C:.1f}x。')
print('    -> 「FVD」这三个字母如果不说清用的是哪种特征，这个数就没有量纲。')
print()
print('  · 场景 C 值得单独说：均值、方差、时间相关性**全部匹配**，只有峰度从 3 变成 1。')
print(f'    FVD(逐帧) 因此完全看不出（{_pf_C:.2f}x）—— 这正是本模块病理二。')
print('    而 FVD(时空) 能看出，机制很具体：|diff| 这个特征把一个**四阶矩**差别')
print('    转成了**一阶矩**差别（两点符号过程的 |Δ| 只取 0 或 2σ，均值与高斯不同）。')
print('    -> 所以病理二的准确表述是「FVD 对**给定特征的**高阶矩免疫」，')
print('       而**选对特征可以把信号搬进前两阶** —— 这就是指标特征工程的全部内容。')
print()
print('  · 覆盖率在场景 C 上只降到 '
      f'{_rC["coverage"]/_rA["coverage"]:.3f}x —— 它不是万能的补丁。')
print('    它的强项是「生成完全没覆盖到某些区域」，而不是「覆盖了但形状不对」。')
print()
print('  · 每一项都带「它对什么免疫」的字段。这不是啰嗦 ——')
print('    本模块的三个病理说明，**不写清免疫范围的指标等于没有量纲**。')